In [ ]:
# Cell 1: Import dependencies and resolve local config paths
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import yaml
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import load_model

notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text())
    pkl_path = Path(cfg.get('datasets', {}).get('rml2016', {}).get('pkl', '/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl'))
else:
    pkl_path = Path('/scratch/rameyjm7/datasets/RML2016/RML2016.10a_dict.pkl')

model_path = project_root / 'models' / 'rml2016' / 'rml2016_lstm_rnn_2024.keras'
print('RML2016 dataset:', pkl_path)
print('RML2016 model:', model_path)
assert pkl_path.exists(), f'Missing dataset: {pkl_path}'
assert model_path.exists(), f'Missing model: {model_path}'
outputs_dir = project_root / 'outputs' / 'rml2016_eval'
outputs_dir.mkdir(parents=True, exist_ok=True)
print('Outputs dir:', outputs_dir)



In [ ]:
# Cell 2: Load dataset and prepare train/test split with IQ+SNR features
with pkl_path.open('rb') as f:
    data = pickle.load(f, encoding='latin1')

X_rows, y_rows = [], []
for (mod, snr), signals in data.items():
    for signal in signals:
        iq = np.vstack([signal[0], signal[1]]).T.astype(np.float32)
        snr_col = np.full((iq.shape[0], 1), snr, dtype=np.float32)
        X_rows.append(np.hstack([iq, snr_col]))
        y_rows.append(mod)

X = np.asarray(X_rows, dtype=np.float32)
y_text = np.asarray(y_rows)

encoder = LabelEncoder()
y = encoder.fit_transform(y_text)

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('X_test shape:', X_test.shape)
print('Classes:', list(encoder.classes_))

In [ ]:
# Cell 3: Run model inference and print metrics
model = load_model(model_path, compile=False)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

acc = accuracy_score(y_test, y_pred)
print(f'RML2016 evaluation accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred, target_names=encoder.classes_, zero_division=0))

In [ ]:
# Cell 4: Plot confusion matrix for RML2016
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues', xticklabels=encoder.classes_, yticklabels=encoder.classes_)
plt.title('RML2016 Confusion Matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Plot and save accuracy vs SNR (overall and per modulation)
import csv

# SNR was appended as feature channel and is constant along the time axis per sample.
snr_test = X_test[:, 0, 2].astype(int)
unique_snrs = np.array(sorted(np.unique(snr_test)), dtype=int)

# Overall accuracy vs SNR
overall_acc = []
for snr in unique_snrs:
    idx = np.where(snr_test == snr)[0]
    overall_acc.append(float(np.mean(y_pred[idx] == y_test[idx])) * 100.0)

# Per-modulation accuracy vs SNR
class_names = list(encoder.classes_)
per_mod_acc = np.full((len(class_names), len(unique_snrs)), np.nan, dtype=np.float32)

for c in range(len(class_names)):
    cls_mask = y_test == c
    for j, snr in enumerate(unique_snrs):
        m = cls_mask & (snr_test == snr)
        if np.any(m):
            per_mod_acc[c, j] = float(np.mean(y_pred[m] == y_test[m])) * 100.0

# Combined figure with bottom-two style charts
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].plot(unique_snrs, overall_acc, marker='o', color='blue')
axes[0].set_title('Recognition Accuracy vs. SNR')
axes[0].set_xlabel('SNR (dB)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].grid(True, alpha=0.4)

for i, mod in enumerate(class_names):
    axes[1].plot(unique_snrs, per_mod_acc[i], marker='o', linewidth=1.4, label=mod)
axes[1].set_title('Accuracy vs. SNR per Modulation Type')
axes[1].set_xlabel('SNR (dB)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, alpha=0.4)
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)

plt.tight_layout()

snr_plots_png = outputs_dir / '40_rml2016_accuracy_vs_snr_plots.png'
plt.savefig(snr_plots_png, dpi=180)
print('Saved SNR plots:', snr_plots_png)
plt.show()

# Save raw series for README/report use
overall_csv = outputs_dir / '40_rml2016_accuracy_vs_snr.csv'
with open(overall_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['snr_db', 'accuracy_percent'])
    for snr, acc_val in zip(unique_snrs, overall_acc):
        writer.writerow([int(snr), f"{acc_val:.6f}"])
print('Saved overall SNR accuracy table:', overall_csv)

per_mod_csv = outputs_dir / '40_rml2016_accuracy_vs_snr_per_modulation.csv'
with open(per_mod_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['modulation'] + [str(int(s)) for s in unique_snrs])
    for i, mod in enumerate(class_names):
        row = [mod] + [
            '' if np.isnan(per_mod_acc[i, j]) else f"{per_mod_acc[i, j]:.6f}"
            for j in range(len(unique_snrs))
        ]
        writer.writerow(row)
print('Saved per-modulation SNR accuracy table:', per_mod_csv)

